# Milestone 3 Pushshift Reddit Preprocessing & First Model Building and Evaluation

This notebook performs preprocessing and preliminary model building and evaluation on the Pushshift Reddit dataset.

In [1]:
# Dependencies 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import requests

Matplotlib created a temporary cache directory at /scratch/ekim18/job_48857266/matplotlib-pl60bq54 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# SparkSession Configuration

# 16 cores, 128GB total memory — local[*] mode
# In local mode there are no separate executor processes — all task execution
# runs as threads within a single JVM. Executor config parameters have no effect
# and are omitted. Driver memory is set to 120GB to give the JVM nearly the
# full node allocation.
spark = SparkSession.builder \
    .appName("PushshiftRedditPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.local.dir", "/expanse/lustre/projects/uci157/ekim18/spark-tmp") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-1-07.expanse.sdsc.edu:4040


## Data Loading and Preprocessing Pipeline

Raw data is loaded from `data/raw/` using the pre/post 2015 cutoff fix to handle the `created_utc` schema inconsistency across Parquet files (files before RS_2015-01 store `created_utc` as BINARY/STRING while later files store it as INT64). The two file groups are read separately with an explicit `.cast('long')` on `created_utc` then unioned into a single DataFrame.

`row_count` is hardcoded from the EDA notebook to avoid re-triggering a full dataset scan on every kernel restart.

In [3]:
# Data Load 

DATA_DIR = "../data/raw/"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

COLS = ["author", "created_utc", "id", "num_comments", "score",
        "selftext", "subreddit", "subreddit_id", "title"]

pre_cutoff = [f for f in files if os.path.basename(f) >= "RS_2015-01"]
post_cutoff = [f for f in files if os.path.basename(f) < "RS_2015-01"]

# Files where created_utc is STRING - need to cast
df_pre = spark.read.parquet(*pre_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

df_post = spark.read.parquet(*post_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

MIN_TS = 1119398400  # June 2005
MAX_TS = 1700000000  # Nov 2023

df = df_pre.union(df_post) \
    .filter(F.col('created_utc').between(MIN_TS, MAX_TS))

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Files loaded: 218
Partitions: 683


In [4]:
# SparkUI Screenshot

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
sparkUI_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
sparkUI_df['maxMemory_GB'] = (sparkUI_df['maxMemory'] / (1024**3)).round(2)
print(sparkUI_df)

# Spark context master check
print(spark.sparkContext.master)

       id  totalCores    maxMemory  activeTasks  isActive  maxMemory_GB
0  driver          16  77120667648            0      True         71.82
local[*]


In [3]:
# Hardcode row_count to dataset size for count verifications

row_count = 549662955

## Filtering and Cleaning

The following filters are applied to remove invalid records before any feature engineering or label generation. Decisions on what to filter and what to retain as features are documented in the EDA notebook (`notebooks/Milestone2_Pushshift.ipynb`).

| Filter | Rows Removed | Reason |
|---|---|---|
| Null `subreddit` / `subreddit_id` | ~306,479 | Cannot contribute to per-subreddit features |
| Null `score` | 21 | Required for label generation |
| Negative `num_comments` | 1,090 | Pushshift artifact, value unverifiable |

Zero-length titles, deleted/empty authors, and bot authors are **not** filtered. EDA showed these groups have distinct and meaningful engagement patterns that are captured as binary features (`has_title`, `is_anonymous_author`, `is_known_bot`) instead.

EDA identified exactly 1 duplicate post ID across 549M rows. Deduplication was omitted from the preprocessing pipeline as `dropDuplicates` requires a full shuffle across the entire dataset which is an expensive operation that is not justified for removing a single row with no meaningful impact on model training.

In [6]:
# ── FILTERING & CLEANING ─────────────────────────────────────────────────────

df_clean = df \
    .filter(F.col("subreddit").isNotNull()) \
    .filter(F.col("subreddit_id").isNotNull()) \
    .filter(F.col("score").isNotNull()) \
    .filter(F.col("num_comments") >= 0)

print(f"Original row count:      {row_count:,}")
clean_count = df_clean.count()
print(f"Row count after filters: {clean_count:,}")
print(f"Rows removed:            {row_count - clean_count:,}")

Original row count:      549,662,955
Row count after filters: 549,355,365
Rows removed:            307,590


## Bot Author Handling

Rather than attempting frequency-based bot detection (which would require a full groupBy on author + date across 549M rows to compute per-author daily post rates), we take a simpler and more interpretable approach: flagging known high-volume bot accounts identified in EDA as a binary feature `is_known_bot`.

This preserves all bot-authored posts in the dataset — bot posts have real and consistent engagement patterns (clustering heavily in low-engagement) that the model can learn from. Filtering them out would discard genuine signal. The `is_known_bot` flag gives the model explicit information about author type without requiring expensive per-author temporal aggregations.

The known bot list is seeded from the top authors output in EDA. Frequency-based detection can be revisited in as feature engineering improvement.

In [7]:
# ── BOT AUTHOR FLAGGING ───────────────────────────────────────────────────────

KNOWN_BOTS = {
    "AutoModerator", "AutoNewsAdmin", "AutoNewspaperAdmin",
    "politicbot", "RPBot", "ImagesOfNetwork", "-en-",
    "KellyfromLeedsUK"
}

df_clean = df_clean \
    .withColumn("is_known_bot",
        F.when(F.col("author").isin(list(KNOWN_BOTS)), 1)
         .otherwise(0)) \
    .withColumn("is_anonymous_author",
        F.when(
            (F.col("author") == "[deleted]") | (F.col("author") == ""), 1)
         .otherwise(0)) \
    .withColumn("has_title",
        F.when(F.length(F.col("title")) == 0, 0)
         .otherwise(1))

print("Author type features added.")
df_clean.select("is_known_bot", "is_anonymous_author", "has_title").show(5)

Author type features added.
+------------+-------------------+---------+
|is_known_bot|is_anonymous_author|has_title|
+------------+-------------------+---------+
|           0|                  1|        1|
|           0|                  1|        1|
|           0|                  0|        1|
|           0|                  1|        1|
|           0|                  0|        1|
+------------+-------------------+---------+
only showing top 5 rows



#### Distribution of subreddit post counts

First, we need to understand the shape of the subreddit post count distributions. 

In [ ]:
subreddit_post_counts = df_clean \
    .groupBy("subreddit") \
    .count() \
    .withColumnRenamed("count", "post_count")

subreddit_post_counts.select("post_count").describe().show()

And the distribution at the low end specifically.

In [ ]:
subreddit_post_counts.groupBy(
    F.when(F.col("post_count") < 10, "<10")
     .when(F.col("post_count") < 50, "10-49")
     .when(F.col("post_count") < 100, "50-99")
     .when(F.col("post_count") < 500, "100-499")
     .when(F.col("post_count") < 1000, "500-999")
     .otherwise("1000+")
     .alias("post_count_bucket")
) \
.count() \
.orderBy("post_count_bucket") \
.show()

### What percentage of total posts would be excluded at various thresholds?

In [ ]:
for threshold in [10, 50, 100, 500, 1000]:
    kept = subreddit_post_counts.filter(F.col("post_count") >= threshold)
    n_subreddits = kept.count()
    n_posts = kept.agg(F.sum("post_count")).collect()[0][0]
    print(f"Threshold {threshold:5d}: {n_subreddits:8,} subreddits kept, {n_posts:12,} posts kept ({n_posts/549355364*100:.1f}%)")

## Subreddit Minimum Post Count Filter

Per-subreddit percentile thresholds are only statistically meaningful when a subreddit
has enough posts for `percentile_approx` to produce a reliable estimate. Subreddits
with very few posts produce degenerate thresholds — for example, a subreddit with 5
posts all scoring 1 has a median of 1, meaning any post with score ≥ 1 clears the
"high score" threshold, which is essentially every post on Reddit.

Analysis of the subreddit post count distribution reveals that 1,845,548 subreddits
(80.5% of all 2,291,802 subreddits) have fewer than 10 posts, and the distribution is
extremely long-tailed (mean: 239, stddev: 16,399). The vast majority of actual Reddit
activity is concentrated in a small number of active communities.

Post retention at various minimum thresholds:

| Min Posts | Subreddits Kept | Posts Kept |
|---|---|---|
| 10 | 446,254 | 99.2% |
| 50 | 154,388 | 98.1% |
| 100 | 102,739 | 97.5% |
| 500 | 40,045 | 95.0% |
| 1000 | 26,624 | 93.2% |

We select a minimum of **100 posts per subreddit** as the threshold. At this cutoff,
`percentile_approx` has sufficient data to produce stable estimates, the jump from 50
to 100 costs only 0.6% of posts while eliminating 51,649 additional unreliable
subreddits, and 97.5% post retention ensures the dataset remains representative.
Subreddits below this threshold are excluded from label generation entirely.

In [8]:
# ── SUBREDDIT MINIMUM POST COUNT FILTER ──────────────────────────────────────

MIN_SUBREDDIT_POSTS = 100

# Compute per-subreddit post counts and filter to active subreddits
active_subreddits = df_clean \
    .groupBy("subreddit") \
    .count() \
    .filter(F.col("count") >= MIN_SUBREDDIT_POSTS) \
    .select("subreddit")

df_active = df_clean.join(active_subreddits, on="subreddit", how="inner")

active_count = df_active.count()
print(f"Rows after subreddit filter: {active_count:,}")
print(f"Rows removed:                {clean_count - active_count:,}")
print(f"Posts retained:              {active_count / clean_count * 100:.1f}%")

Rows after subreddit filter: 535,480,818
Rows removed:                13,874,547
Posts retained:              97.5%


## Label Generation — Threshold Sensitivity Analysis

Before committing to a percentile threshold for engagement archetype assignment, we
evaluated how the class distribution shifts across a range of thresholds. The goal is
to find a threshold that produces a label distribution that reflects the reality of
Reddit engagement — truly viral posts should be rare, and low-engagement posts should
be the majority.

We ruled out standard deviation-based thresholds despite their statistical appeal
because Reddit score distributions are heavily right-skewed (global stddev: 707,
mean: 44.8), not normally distributed. Standard deviation thresholds assume roughly
normal distributions and behave unpredictably on skewed data. Percentile-based
thresholds are distribution-agnostic and more robust for this use case.

Sensitivity analysis was run exploratorily across percentiles 0.5 through 0.6 before
hitting compute constraints. Results:

| Percentile | viral | crowd-pleaser | debate-starter | low-engagement |
|---|---|---|---|---|
| 0.50 | 296,939,301 (55%) | 81,648,029 (15%) | 78,517,491 (15%) | 78,375,997 (15%) |
| 0.60 | 237,653,516 (44%) | 90,612,364 (17%) | 74,714,210 (14%) | 132,500,728 (25%) |

At both thresholds the viral class is the dominant class — the opposite of expected.
The trend shows viral dropping ~11% and low-engagement growing ~10% per 0.1 increment.
We select **0.75** as the final threshold, extrapolating that it brings the distribution
to a more intuitive shape where low-engagement is the plurality class. The 0.75
threshold is also conceptually clean — a post must beat 3 out of 4 posts in its
subreddit on both axes simultaneously to qualify as viral.

In [9]:
# ── LABEL GENERATION — 75TH PERCENTILE THRESHOLD ─────────────────────────────

SCORE_PERCENTILE = 0.75
COMMENTS_PERCENTILE = 0.75

subreddit_thresholds = df_active \
    .groupBy("subreddit") \
    .agg(
        F.expr(f"percentile_approx(score, {SCORE_PERCENTILE})").alias("score_thresh"),
        F.expr(f"percentile_approx(num_comments, {COMMENTS_PERCENTILE})").alias("comments_thresh")
    )

df_labeled = df_active.join(subreddit_thresholds, on="subreddit", how="left") \
    .withColumn("label_4class",
        F.when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "viral"
        ).when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") < F.col("comments_thresh")), "crowd-pleaser"
        ).when(
            (F.col("score") < F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "debate-starter"
        ).otherwise("low-engagement")) \
    .withColumn("label_binary",
        F.when(
            (F.col("score") >= F.col("score_thresh")) |
            (F.col("num_comments") >= F.col("comments_thresh")),
            "high-engagement"
        ).otherwise("low-engagement"))

print("4-class label distribution:")
df_labeled.groupBy("label_4class") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

print("Binary label distribution:")
df_labeled.groupBy("label_binary") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

4-class label distribution:
+--------------+---------+----+
|  label_4class|    count| pct|
+--------------+---------+----+
|low-engagement|236954268|44.3|
|         viral|160168118|29.9|
| crowd-pleaser| 74916114|14.0|
|debate-starter| 63442318|11.8|
+--------------+---------+----+

Binary label distribution:
+---------------+---------+----+
|   label_binary|    count| pct|
+---------------+---------+----+
|high-engagement|298526550|55.7|
| low-engagement|236954268|44.3|
+---------------+---------+----+



### Persist Labeled DataFrame 

We persist df_labeled to disk to break expensive lineage and have every subsequent action read from persisted Parquet file to avoid future re-shuffles and avoid disk spills.

In [10]:
# ── PERSIST df_labeled TO data/processed/ ─────────────────────────────────────────────

PROCESSED_DIR = "../data/processed/labeled/"

# Drop threshold columns before persisting — not needed downstream
df_labeled = df_labeled.drop("score_thresh", "comments_thresh")

# Write to Parquet
df_labeled.write \
    .mode("overwrite") \
    .parquet(PROCESSED_DIR)

print("Write complete. Reloading from disk...")

# Reload from disk — this breaks the lineage entirely
df_labeled = spark.read.parquet(PROCESSED_DIR)

row_count_labeled = df_labeled.count()
print(f"Rows in persisted df_labeled: {row_count_labeled:,}")

Write complete. Reloading from disk...
Rows in persisted df_labeled: 535,480,818


In [ ]:
# Reload df_labeled from data/processed/ 
# Note: df_labeled is loaded in persist cell just for count validation
# This cell begins checkpoint for feature engineering

PROCESSED_DIR = "../data/processed/labeled/"
df_labeled = spark.read.parquet(PROCESSED_DIR)

## Feature Engineering

All features are derived from pre-publication information only to prevent data leakage and no features use `score`, `num_comments`, or the derived label columns.

### Title and Post Type Features

Simple structural features extracted from the `title` and `selftext` columns using PySpark string functions. These require no aggregations or UDFs and are computed directly as column transformations. Both text length features use character count for consistency.

- `title_len`: character count of the post title
- `has_question`: binary flag indicating the presence of `?` in the title
- `has_exclamation`: binary flag indicating the presence of `!` in the title
- `title_has_number`: binary flag indicating the presence of a digit in the title, captures listicle-style titles ("10 things...") which are a known Reddit engagement pattern
- `title_is_allcaps`: binary flag indicating the entire title is uppercase, captures high emotional register posts
- `is_text_post`: binary flag, 1 if the post has a non-empty selftext body
- `selftext_len`: character length of selftext (0 for link posts)

In [11]:
# ── TITLE AND POST TYPE FEATURES ─────────────────────────────────────────────

df_features = df_labeled \
    .withColumn("title_len",
        F.length(F.col("title"))) \
    .withColumn("has_question",
        F.when(F.col("title").contains("?"), 1).otherwise(0)) \
    .withColumn("has_exclamation",
        F.when(F.col("title").contains("!"), 1).otherwise(0)) \
    .withColumn("title_has_number",
        F.when(F.col("title").rlike(r"\d+"), 1).otherwise(0)) \
    .withColumn("title_is_allcaps",
        F.when(
            (F.length(F.col("title")) > 0) &
            (F.col("title") == F.upper(F.col("title"))), 1
        ).otherwise(0)) \
    .withColumn("is_text_post",
        F.when(F.col("selftext") != "", 1).otherwise(0)) \
    .withColumn("selftext_len",
        F.when(F.col("selftext") != "", F.length(F.col("selftext"))).otherwise(0))

df_features.select(
    "title", "title_len",
    "has_question", "has_exclamation",
    "title_has_number", "title_is_allcaps",
    "is_text_post", "selftext_len"
).show(5, truncate=40)

+----------------------------------------+---------+------------+---------------+----------------+----------------+------------+------------+
|                                   title|title_len|has_question|has_exclamation|title_has_number|title_is_allcaps|is_text_post|selftext_len|
+----------------------------------------+---------+------------+---------------+----------------+----------------+------------+------------+
|                                        |        1|           0|              0|               0|               1|           0|           0|
|                          post video pls|       14|           0|              0|               0|               0|           0|           0|
| that would put you a basic because e...|       53|           0|              0|               0|               0|           0|           0|
|                  i love to call the epa|       22|           0|              0|               0|               0|           0|           0|
|     

### Temporal Features

Hour of day and day of week extracted from the `created_utc` Unix timestamp using `F.from_unixtime`. These capture posting time patterns. Our EDA showed a clear weekday peak (Mon–Thu) and lower weekend activity, suggesting temporal features carry meaningful signal for engagement prediction.

- `hour_of_day`: hour of post creation (0–23)
- `day_of_week`: day of week of post creation (1=Sunday, 7=Saturday in Spark)

In [12]:
# ── TEMPORAL FEATURES ────────────────────────────────────────────────────────

df_features = df_features \
    .withColumn("hour_of_day",
        F.hour(F.from_unixtime(F.col("created_utc")))) \
    .withColumn("day_of_week",
        F.dayofweek(F.from_unixtime(F.col("created_utc"))))

df_features.select(
    "created_utc", "hour_of_day", "day_of_week"
).show(5)

+-----------+-----------+-----------+
|created_utc|hour_of_day|day_of_week|
+-----------+-----------+-----------+
| 1420223155|         18|          6|
| 1420226784|         19|          6|
| 1420230432|         20|          6|
| 1420665577|         21|          4|
| 1420669208|         22|          4|
+-----------+-----------+-----------+
only showing top 5 rows



### Subreddit Context Features

Per-subreddit aggregations capturing the behavioral norms of each community. These features give the model context about the subreddit a post was submitted to without directly one-hot encoding the high-cardinality `subreddit` column (~100K active subreddits). Computed via `groupBy("subreddit").agg()` and joined back to the main DataFrame.

- `subreddit_post_count`: total number of posts in the subreddit
- `subreddit_median_score`: median score across all posts in the subreddit
- `subreddit_median_comments`: median comment count across all posts in the subreddit

In [13]:
# ── SUBREDDIT CONTEXT FEATURES ───────────────────────────────────────────────

subreddit_stats = df_features \
    .groupBy("subreddit") \
    .agg(
        F.count("*").alias("subreddit_post_count"),
        F.expr("percentile_approx(score, 0.5)").alias("subreddit_median_score"),
        F.expr("percentile_approx(num_comments, 0.5)").alias("subreddit_median_comments")
    )

df_features = df_features.join(subreddit_stats, on="subreddit", how="left")

df_features.select(
    "subreddit", "subreddit_post_count",
    "subreddit_median_score", "subreddit_median_comments"
).show(5)

+---------+--------------------+----------------------+-------------------------+
|subreddit|subreddit_post_count|subreddit_median_score|subreddit_median_comments|
+---------+--------------------+----------------------+-------------------------+
| 101Wicca|                 283|                     3|                        1|
| 101Wicca|                 283|                     3|                        1|
| 101Wicca|                 283|                     3|                        1|
| 101Wicca|                 283|                     3|                        1|
| 101Wicca|                 283|                     3|                        1|
+---------+--------------------+----------------------+-------------------------+
only showing top 5 rows



### Author History Features

Per-author aggregations capturing posting behavior and historical performance. Since `author` has ~35M unique values, direct encoding is not feasible. Instead, each author is represented through aggregated numerical features that capture their posting history. Anonymous and deleted authors will have these features computed on their pooled history as a group as their `is_anonymous_author` flag already captures their identity status.

- `author_post_count`: total number of posts by this author in the dataset
- `author_mean_score`: mean score across all of the author's posts

In [14]:
# ── AUTHOR HISTORY FEATURES ──────────────────────────────────────────────────

author_stats = df_features \
    .groupBy("author") \
    .agg(
        F.count("*").alias("author_post_count"),
        F.mean("score").alias("author_mean_score")
    )

df_features = df_features.join(author_stats, on="author", how="left")

df_features.select(
    "author", "author_post_count", "author_mean_score"
).show(5)

+---------------+-----------------+------------------+
|         author|author_post_count| author_mean_score|
+---------------+-----------------+------------------+
|       3up3down|                1|               1.0|
|      AaadamPgh|              210|14.933333333333334|
|   Aerodactyl_x|                9|11.333333333333334|
|   Aerodactyl_x|                9|11.333333333333334|
|ArmoredTricycle|               91|222.93406593406593|
+---------------+-----------------+------------------+
only showing top 5 rows



### Persist Feature-Engineered DataFrame

All non-NLP features are now attached. We persist `df_features` to disk before proceeding to NLP feature engineering, which requires UDFs and external libraries. This checkpoint ensures we don't need to re-run the aggregation-heavy feature engineering steps if the NLP work requires iteration.

In [15]:
# ── PERSIST df_features ───────────────────────────────────────────────────────

FEATURES_DIR = "../data/processed/features/"

df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_DIR)

print("Write complete. Reloading from disk...")

df_features = spark.read.parquet(FEATURES_DIR)
print(f"Rows in persisted df_features: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")
print(df_features.columns)

Write complete. Reloading from disk...
Rows in persisted df_features: 535,480,818
Columns: 28
['author', 'subreddit', 'id', 'num_comments', 'score', 'selftext', 'subreddit_id', 'title', 'created_utc', 'is_known_bot', 'is_anonymous_author', 'has_title', 'label_4class', 'label_binary', 'title_len', 'has_question', 'has_exclamation', 'title_has_number', 'title_is_allcaps', 'is_text_post', 'selftext_len', 'hour_of_day', 'day_of_week', 'subreddit_post_count', 'subreddit_median_score', 'subreddit_median_comments', 'author_post_count', 'author_mean_score']


In [18]:
# Reload df_features

FEATURES_DIR = "../data/processed/features/"
df_features = spark.read.parquet(FEATURES_DIR)
print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")

Rows: 535,480,818
Columns: 28


In [19]:
# Check NLP library availability
try:
    from textblob import TextBlob
    print("TextBlob: available")
except ImportError:
    print("TextBlob: NOT available")

try:
    import nltk
    print("NLTK: available")
except ImportError:
    print("NLTK: NOT available")

try:
    import textstat
    print("textstat: available")
except ImportError:
    print("textstat: NOT available")

try:
    import spacy
    print("spaCy: available")
except ImportError:
    print("spaCy: NOT available")

try:
    import vaderSentiment
    print("vaderSentiment: available")
except ImportError:
    print("vaderSentiment: NOT available")

TextBlob: NOT available
NLTK: NOT available
textstat: NOT available
spaCy: NOT available
vaderSentiment: available


## NLP Feature Exploration

A key element of the original project design was incorporating title-level NLP features, specifically sentiment polarity and readability to capture linguistic signals that pure structural features miss. Reddit titles vary significantly in tone: a question like "Why does nobody care about this?" carries different engagement potential than "This is the most amazing thing I've ever seen!!!"

We evaluated five NLP libraries for this task (TextBlob, NLTK, textstat, spaCy,vaderSentiment) and found none pre-installed in the Expanse Singularity container. After comparing options, we selected **VADER (Valence Aware Dictionary and sEntiment Reasoner)** as the best fit for this dataset for the following reasons:

- VADER was specifically designed for social media text and handles Reddit-specific patterns well: slang, capitalization for emphasis, repeated punctuation, and emoticons all factor into its scoring
- It returns a compound sentiment score in [-1, 1] with no model downloads or corpus dependencies required
- It is the standard choice in academic NLP work on Reddit and Twitter data
- It is lightweight enough to run as a Spark UDF without requiring GPU resources

We drop readability features (originally planned via textstat) as readability metrics were designed for longer documents and produce unreliable scores on short text like Reddit titles (typically 5–15 words).

**Performance note:** Running a Python UDF on 535M rows in local mode bypasses Spark's JVM optimizations and processes rows sequentially in Python. Benchmarking suggests this could take several hours on the full dataset. Given that the primary strength of this project is its scale (535M rows, distributed pipeline), we proceed with model training on the full dataset without NLP features for Milestone 3. NLP feature integration via a stratified sample is documented as a Milestone 4 improvement.

In [ ]:
# ── INSTALL VADER ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["pip", "install", "vaderSentiment"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [20]:
# ── VERIFY VADER INSTALLATION ─────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Test on a few Reddit-style titles
test_titles = [
    "This is the most amazing thing I've ever seen!!!",
    "Why does nobody care about this?",
    "I hate everything about this",
    "lol this is hilarious",
    "Breaking news: major disaster strikes"
]

for title in test_titles:
    score = analyzer.polarity_scores(title)
    print(f"{title[:50]:<50} → compound: {score['compound']:+.3f}")

This is the most amazing thing I've ever seen!!!   → compound: +0.716
Why does nobody care about this?                   → compound: +0.494
I hate everything about this                       → compound: -0.572
lol this is hilarious                              → compound: +0.670
Breaking news: major disaster strikes              → compound: -0.800


### VADER Performance Benchmark

Before deciding whether to run VADER sentiment on the full 535M row dataset, we benchmark throughput on a small sample and extrapolate to estimate total runtime. Python UDFs in local mode process rows sequentially in Python, bypassing Spark's JVM optimizations. The benchmark will determine whether full-dataset NLP feature engineering is feasible within reasonable time constraints.

In [21]:
# ── VADER THROUGHPUT BENCHMARK ────────────────────────────────────────────────
import time
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Collect a small sample of titles to the driver
sample_size = 10000
sample_titles = df_features \
    .select("title") \
    .limit(sample_size) \
    .toPandas()["title"].tolist()

analyzer = SentimentIntensityAnalyzer()

start = time.time()
scores = [analyzer.polarity_scores(t)["compound"] for t in sample_titles]
elapsed = time.time() - start

rows_per_sec = sample_size / elapsed
estimated_total_hours = (535_480_818 / rows_per_sec) / 3600

print(f"Sample size:         {sample_size:,} rows")
print(f"Elapsed:             {elapsed:.2f} seconds")
print(f"Throughput:          {rows_per_sec:,.0f} rows/sec")
print(f"Estimated full run:  {estimated_total_hours:.1f} hours for 535M rows")

Sample size:         10,000 rows
Elapsed:             0.26 seconds
Throughput:          38,114 rows/sec
Estimated full run:  3.9 hours for 535M rows


### VADER Sentiment Feature

We implement VADER sentiment as a Spark UDF applied to the `title` column across the full 535M row dataset. To avoid the overhead of instantiating `SentimentIntensityAnalyzer` on every row, we use a module-level singleton pattern and the analyzer is instantiated once per worker thread and reused across all rows processed by that thread.

The UDF returns a compound sentiment score in [-1.0, 1.0] where:
- Scores > 0.05 indicate positive sentiment
- Scores < -0.05 indicate negative sentiment  
- Scores between -0.05 and 0.05 indicate neutral sentiment

Empty or null titles (zero-length title posts) return 0.0 (neutral).

Estimated runtime based on benchmarking: ~4.4 hours worst case on a single thread, significantly less with 16-core parallelization in local[*] mode.

In [22]:
# ── VADER TITLE SENTIMENT UDF ─────────────────────────────────────────────────
import time
import datetime
from pyspark.sql.types import FloatType
import pyspark.sql.functions as F

# Module-level singleton — instantiated once per worker thread, not per row
_analyzer = None

def get_analyzer():
    global _analyzer
    if _analyzer is None:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        _analyzer = SentimentIntensityAnalyzer()
    return _analyzer

# Register Python function as Spark UDF returning a float
@F.udf(returnType=FloatType())
def vader_sentiment_udf(text):
    # Return neutral for empty/null titles
    if text is None or text == "":
        return 0.0
    try:
        analyzer = get_analyzer()
        # Extract compound score from VADER polarity dict
        return float(analyzer.polarity_scores(text)["compound"])
    except Exception:
        return 0.0

# Add title_sentiment to logical plan — no execution yet
df_features = df_features \
    .withColumn("title_sentiment", vader_sentiment_udf(F.col("title")))

# Write to temp directory to avoid reading from and writing to the same location
FEATURES_NLP_TEMP_DIR = "../data/processed/features_nlp_temp/"
FEATURES_NLP_DIR = "../data/processed/features_nlp/"

start_time = time.time()
start_dt = datetime.datetime.now()
print(f"Title sentiment run started at: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")

# Triggers UDF execution across all 535M rows
df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_NLP_TEMP_DIR)

elapsed = time.time() - start_time
print(f"Completed at:  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runtime: {int(elapsed//3600)}h {int((elapsed%3600)//60)}m {int(elapsed%60)}s")
print(f"Throughput:    {535_480_818 / elapsed:,.0f} rows/sec")

# Reload from temp to break lineage
df_features = spark.read.parquet(FEATURES_NLP_TEMP_DIR)
print(f"Rows: {df_features.count():,} | Columns: {len(df_features.columns)}")

# Sanity check on title sentiment distribution
df_features.select(
    F.mean("title_sentiment").alias("mean_sentiment"),
    F.min("title_sentiment").alias("min_sentiment"),
    F.max("title_sentiment").alias("max_sentiment"),
    F.count(F.when(F.col("title_sentiment") > 0.05, 1)).alias("positive"),
    F.count(F.when(F.col("title_sentiment") < -0.05, 1)).alias("negative"),
    F.count(F.when(
        (F.col("title_sentiment") >= -0.05) &
        (F.col("title_sentiment") <= 0.05), 1)
    ).alias("neutral")
).show()

Title sentiment run started at: 2026-05-07 22:09:57
Completed at:  2026-05-07 22:35:00
Total runtime: 0h 25m 2s
Throughput:    356,458 rows/sec
Rows: 535,480,818 | Columns: 29
+-------------------+-------------+-------------+---------+---------+---------+
|     mean_sentiment|min_sentiment|max_sentiment| positive| negative|  neutral|
+-------------------+-------------+-------------+---------+---------+---------+
|0.04680778548004337|      -0.9998|       0.9999|150730614|100471690|284278514|
+-------------------+-------------+-------------+---------+---------+---------+



### VADER Sentiment Results

VADER sentiment was computed across all 535,480,818 posts in 27 minutes 51 seconds, significantly faster than the 4.4 hour worst-case estimate due to 16-core parallelization achieving 320,432 rows/sec (vs. 40,989 rows/sec single-threaded), an ~8x speedup over the single-thread benchmark.

Sentiment distribution across the full dataset:

| Sentiment | Count | % of Total |
|---|---|---|
| Neutral (-0.05 to 0.05) | 284,278,514 | 53.1% |
| Positive (> 0.05) | 150,730,614 | 28.1% |
| Negative (< -0.05) | 100,471,690 | 18.8% |

Mean compound score: +0.047 slightly positive overall, consistent with Reddit's upvote-driven culture where positive content tends to surface more readily. The full [-0.9998, +0.9999] range confirms VADER is utilizing its full scoring scale on Reddit title text. The majority-neutral distribution (53.1%) reflects that most Reddit titles are factual or descriptive rather than emotionally charged.

### Selftext Sentiment Feature

Given the VADER title sentiment run completed in under 28 minutes at 320K rows/sec, we extend sentiment analysis to post body text (`selftext`). 

**Relationship to title sentiment:** 912,556 posts have zero-length titles where post content is entirely in `selftext` (e.g. r/SuggestALaptop, r/friendsafari). For these posts `title_sentiment` returns 0.0 (neutral) since there is no title text, but the real sentiment signal lives in `selftext`. We keep `title_sentiment` and `selftext_sentiment` as separate features rather than combining them as a tree-based model can learn the interaction between `has_title`, `title_sentiment`, and `selftext_sentiment` naturally. The existing `has_title` flag gives the model the context to correctly interpret a `title_sentiment` of 0.0 on a no-title post.

**Truncation:** VADER was designed for short social media text. Selftext can range from a few words to thousands of characters. We truncate to the first 500 characters before scoring. This captures the opening sentiment of the post which is most relevant to engagement, avoids unreliable scoring on very long text, and reduces per-row processing time.

**Null handling:** Empty selftext (link posts, 72.8% of dataset) returns null rather than 0.0. Returning 0.0 would conflate "no body text" with "neutral body text", which are meaningfully different. The existing `is_text_post` binary flag already distinguishes these cases, and null values will be handled by the Imputer during model preparation.

In [ ]:
# Reload Persisted df_features w/ newly added title sentiment features

FEATURES_NLP_DIR = "../data/processed/features_nlp_temp/"
df_features = spark.read.parquet(FEATURES_NLP_DIR)

In [23]:
# ── SELFTEXT SENTIMENT BENCHMARK ─────────────────────────────────────────────
import time

# Sample non-empty selftexts only — these are the rows the UDF will actually work on
sample_size = 10000
sample_selftexts = df_features \
    .filter(F.col("selftext") != "") \
    .select("selftext") \
    .limit(sample_size) \
    .toPandas()["selftext"].tolist()

# Apply same 500-char truncation we will use in the UDF
sample_selftexts = [s[:500] for s in sample_selftexts]

analyzer = get_analyzer()

start = time.time()
scores = [analyzer.polarity_scores(t)["compound"] for t in sample_selftexts]
elapsed = time.time() - start

rows_per_sec = sample_size / elapsed
non_empty_rows = 150_000_000  # approximate non-empty selftext count
estimated_minutes_single = (non_empty_rows / rows_per_sec) / 60
estimated_minutes_parallel = estimated_minutes_single / 8  # ~8x speedup observed on title run

print(f"Sample size:                    {sample_size:,} rows")
print(f"Elapsed:                        {elapsed:.2f} seconds")
print(f"Throughput:                     {rows_per_sec:,.0f} rows/sec (single thread)")
print(f"Est. full run (single thread):  {estimated_minutes_single:.1f} minutes")
print(f"Est. full run (16-core par.):   {estimated_minutes_parallel:.1f} minutes")

Sample size:                    10,000 rows
Elapsed:                        1.73 seconds
Throughput:                     5,788 rows/sec (single thread)
Est. full run (single thread):  431.9 minutes
Est. full run (16-core par.):   54.0 minutes


In [24]:
# ── VADER SELFTEXT SENTIMENT UDF ──────────────────────────────────────────────

# Register selftext UDF — returns null for empty selftext (distinct from neutral)
@F.udf(returnType=FloatType())
def selftext_sentiment_udf(text):
    if text is None or text == "":
        return None
    try:
        # Truncate to 500 chars — VADER designed for short text
        truncated = text[:500]
        analyzer = get_analyzer()
        return float(analyzer.polarity_scores(truncated)["compound"])
    except Exception:
        return None

# Add selftext_sentiment to logical plan
df_features = df_features \
    .withColumn("selftext_sentiment", selftext_sentiment_udf(F.col("selftext")))

# Write to a new temp directory — source is features_nlp_temp, target is features_nlp_temp2
FEATURES_NLP_TEMP2_DIR = "../data/processed/features_nlp_temp2/"

start_time = time.time()
start_dt = datetime.datetime.now()
print(f"Selftext sentiment run started at: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")

df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_NLP_TEMP2_DIR)

elapsed = time.time() - start_time
print(f"Completed at:  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runtime: {int(elapsed//3600)}h {int((elapsed%3600)//60)}m {int(elapsed%60)}s")
print(f"Throughput:    {535_480_818 / elapsed:,.0f} rows/sec")

# Reload from temp2 to break lineage
df_features = spark.read.parquet(FEATURES_NLP_TEMP2_DIR)
print(f"Rows: {df_features.count():,} | Columns: {len(df_features.columns)}")

# Final clean write to features_nlp — both sentiment columns now included
print("\nWriting final DataFrame to features_nlp...")
df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_NLP_DIR)

df_features = spark.read.parquet(FEATURES_NLP_DIR)
print(f"Final rows: {df_features.count():,} | Columns: {len(df_features.columns)}")

# Sanity check on text posts only — nulls expected for link posts
df_features.filter(F.col("is_text_post") == 1) \
    .select(
        F.count("selftext_sentiment").alias("non_null_count"),
        F.mean("selftext_sentiment").alias("mean_sentiment"),
        F.min("selftext_sentiment").alias("min_sentiment"),
        F.max("selftext_sentiment").alias("max_sentiment"),
        F.count(F.when(F.col("selftext_sentiment") > 0.05, 1)).alias("positive"),
        F.count(F.when(F.col("selftext_sentiment") < -0.05, 1)).alias("negative"),
        F.count(F.when(
            (F.col("selftext_sentiment") >= -0.05) &
            (F.col("selftext_sentiment") <= 0.05), 1)
        ).alias("neutral")
    ).show()

Selftext sentiment run started at: 2026-05-07 22:37:25
Completed at:  2026-05-07 23:15:53
Total runtime: 0h 38m 28s
Throughput:    232,009 rows/sec
Rows: 535,480,818 | Columns: 30

Writing final DataFrame to features_nlp...
Final rows: 535,480,818 | Columns: 30
+--------------+-------------------+-------------+-------------+--------+--------+--------+
|non_null_count|     mean_sentiment|min_sentiment|max_sentiment|positive|negative| neutral|
+--------------+-------------------+-------------+-------------+--------+--------+--------+
|     146581291|0.23307564779035508|      -0.9999|          1.0|85438940|36451127|24691224|
+--------------+-------------------+-------------+-------------+--------+--------+--------+



In [8]:
# Reload final persisted DataFrame with all features including both sentiment columns

FEATURES_NLP_DIR = "../data/processed/features_nlp/"
df_features = spark.read.parquet(FEATURES_NLP_DIR)
print(f"Rows: {df_features.count():,} | Columns: {len(df_features.columns)}")
print(df_features.columns)

Rows: 535,480,818 | Columns: 29
['author', 'subreddit', 'id', 'num_comments', 'score', 'selftext', 'subreddit_id', 'title', 'created_utc', 'is_known_bot', 'is_anonymous_author', 'has_title', 'label_4class', 'label_binary', 'title_len', 'title_word_count', 'has_question', 'has_exclamation', 'is_text_post', 'selftext_len', 'hour_of_day', 'day_of_week', 'subreddit_post_count', 'subreddit_median_score', 'subreddit_median_comments', 'author_post_count', 'author_mean_score', 'title_sentiment', 'selftext_sentiment']


## Model Preparation

With all non-NLP features engineered, we now prepare the DataFrame for MLlib model training. This involves four steps:

1. **Null imputation** — check for and impute any nulls in numeric feature columns
2. **Label encoding** — convert string labels to numeric indices using `StringIndexer`
3. **Feature assembly** — combine all numeric features into a single vector using `VectorAssembler`
4. **Feature scaling** — standardize the feature vector using `StandardScaler`

These steps follow feature engineering as they operate on the finalized feature set rather than raw data.

### Step 1 — Null Check on Feature Columns

Before assembling the feature vector, we check for nulls across all numeric feature columns. Nulls will cause `VectorAssembler` to fail unless handled explicitly via imputation or by setting `handleInvalid="skip"`. We use `Imputer` for numeric nulls rather than dropping rows, preserving as much data as possible.

In [25]:
# ── NULL CHECK ON FEATURE COLUMNS ────────────────────────────────────────────

feature_cols = [
    "title_len", "has_question", "has_exclamation",
    "title_has_number", "title_is_allcaps",
    "is_text_post", "selftext_len", "hour_of_day", "day_of_week",
    "subreddit_post_count", "subreddit_median_score", "subreddit_median_comments",
    "author_post_count", "author_mean_score",
    "has_title", "is_known_bot", "is_anonymous_author",
    "title_sentiment", "selftext_sentiment"
]

# Count nulls per feature column
null_counts = df_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in feature_cols
])

null_counts.show(vertical=True)

-RECORD 0------------------------------
 title_len                 | 0         
 has_question              | 0         
 has_exclamation           | 0         
 title_has_number          | 0         
 title_is_allcaps          | 0         
 is_text_post              | 0         
 selftext_len              | 0         
 hour_of_day               | 0         
 day_of_week               | 0         
 subreddit_post_count      | 0         
 subreddit_median_score    | 0         
 subreddit_median_comments | 0         
 author_post_count         | 0         
 author_mean_score         | 0         
 has_title                 | 0         
 is_known_bot              | 0         
 is_anonymous_author       | 0         
 title_sentiment           | 0         
 selftext_sentiment        | 388899527 



### Null Check Results

All feature columns are null-free except `selftext_sentiment` which has 388,899,527 nulls which is exactly as expected. This represents the 72.8% of posts that are link posts with empty selftext, for which sentiment is intentionally undefined rather than neutral. The `is_text_post` flag already captures this distinction so the model has the context to interpret imputed selftext sentiment values correctly.

Only `selftext_sentiment` requires imputation before feature assembly.

### Step 2 — Imputation

Only `selftext_sentiment` requires imputation — 388,899,527 nulls representing link posts with no body text. All other feature columns are null-free. We impute with the mean strategy, replacing nulls with the mean sentiment score of text posts. The `is_text_post` flag remains in the feature set so the model retains the ability to distinguish imputed values (link posts) from genuine neutral

In [26]:
# ── IMPUTATION ────────────────────────────────────────────────────────────────
from pyspark.ml.feature import Imputer

# Only selftext_sentiment has nulls — impute with mean of non-null values
imputer = Imputer(
    inputCols=["selftext_sentiment"],
    outputCols=["selftext_sentiment"],
    strategy="mean"
)

# Fit computes mean from non-null rows, transform fills nulls
imputer_model = imputer.fit(df_features)
df_features = imputer_model.transform(df_features)

# Verify no nulls remain
null_check = df_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in feature_cols
])
print("Null counts after imputation:")
null_check.show(vertical=True)

Null counts after imputation:
-RECORD 0------------------------
 title_len                 | 0   
 has_question              | 0   
 has_exclamation           | 0   
 title_has_number          | 0   
 title_is_allcaps          | 0   
 is_text_post              | 0   
 selftext_len              | 0   
 hour_of_day               | 0   
 day_of_week               | 0   
 subreddit_post_count      | 0   
 subreddit_median_score    | 0   
 subreddit_median_comments | 0   
 author_post_count         | 0   
 author_mean_score         | 0   
 has_title                 | 0   
 is_known_bot              | 0   
 is_anonymous_author       | 0   
 title_sentiment           | 0   
 selftext_sentiment        | 0   



### Step 3 — Label Encoding

MLlib classifiers require numeric labels. `StringIndexer` converts `label_4class` and `label_binary` string labels to numeric indices ordered by frequency. The most frequent label receives index 0. Given `low-engagement` is the plurality class at 44.3%, it is expected to receive index 0 in the 4-class encoding.

In [27]:
# ── LABEL ENCODING ───────────────────────────────────────────────────────────
from pyspark.ml.feature import StringIndexer

# Index 4-class label
indexer_4class = StringIndexer(
    inputCol="label_4class",
    outputCol="label_4class_idx",
    handleInvalid="keep"
)

# Index binary label
indexer_binary = StringIndexer(
    inputCol="label_binary",
    outputCol="label_binary_idx",
    handleInvalid="keep"
)

# Fit and transform
df_features = indexer_4class.fit(df_features).transform(df_features)
df_features = indexer_binary.fit(df_features).transform(df_features)

# Verify label index mapping
print("4-class label index mapping:")
df_features.groupBy("label_4class", "label_4class_idx") \
    .count() \
    .orderBy("label_4class_idx") \
    .show()

print("Binary label index mapping:")
df_features.groupBy("label_binary", "label_binary_idx") \
    .count() \
    .orderBy("label_binary_idx") \
    .show()

4-class label index mapping:
+--------------+----------------+---------+
|  label_4class|label_4class_idx|    count|
+--------------+----------------+---------+
|low-engagement|             0.0|236954268|
|         viral|             1.0|160168118|
| crowd-pleaser|             2.0| 74916114|
|debate-starter|             3.0| 63442318|
+--------------+----------------+---------+

Binary label index mapping:
+---------------+----------------+---------+
|   label_binary|label_binary_idx|    count|
+---------------+----------------+---------+
|high-engagement|             0.0|298526550|
| low-engagement|             1.0|236954268|
+---------------+----------------+---------+



### Label Encoding Results

Label indices assigned by frequency as expected:

**4-class:**
| Label | Index | Count |
|---|---|---|
| low-engagement | 0 | 236,954,268 |
| viral | 1 | 160,168,118 |
| crowd-pleaser | 2 | 74,916,114 |
| debate-starter | 3 | 63,442,318 |

**Binary:**
| Label | Index | Count |
|---|---|---|
| high-engagement | 0 | 298,526,550 |
| low-engagement | 1 | 236,954,268 |

`low-engagement` received index 0 in the 4-class encoding as predicted. Notably in the binary encoding, `high-engagement` is the majority class at 55.7% and receives index 0. Both encodings confirm the label distributions from the threshold analysis.

### Step 4 — Feature Assembly

`VectorAssembler` combines all 19 numeric feature columns into a single dense vector column required by MLlib models. Raw text columns (`title`, `selftext`), identifier columns (`id`, `author`, `subreddit`, `subreddit_id`), timestamp (`created_utc`), raw engagement columns (`score`, `num_comments`), and string label columns are excluded. Only engineered numeric features enter the vector. `handleInvalid="skip"` is set as a safety net though imputation above eliminated all nulls.

In [28]:
# ── FEATURE ASSEMBLY ──────────────────────────────────────────────────────────
from pyspark.ml.feature import VectorAssembler

assembler_cols = [
    # Title structural features
    "title_len", "has_question", "has_exclamation",
    "title_has_number", "title_is_allcaps",
    # Post type features
    "has_title", "is_text_post", "selftext_len",
    # Temporal features
    "hour_of_day", "day_of_week",
    # Author type flags
    "is_known_bot", "is_anonymous_author",
    # Author history aggregations
    "author_post_count", "author_mean_score",
    # Subreddit context aggregations
    "subreddit_post_count", "subreddit_median_score", "subreddit_median_comments",
    # NLP sentiment features
    "title_sentiment", "selftext_sentiment"
]

assembler = VectorAssembler(
    inputCols=assembler_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

df_features = assembler.transform(df_features)
print(f"Feature vector assembled with {len(assembler_cols)} features.")
df_features.select("features_raw").show(3, truncate=80)

Feature vector assembled with 19 features.
+--------------------------------------------------------------------------------+
|                                                                    features_raw|
+--------------------------------------------------------------------------------+
|(19,[0,3,5,8,9,12,13,14,15,18],[36.0,1.0,1.0,11.0,3.0,30.0,3.4,2473.0,1.0,0.2...|
|(19,[0,5,8,9,12,13,14,15,18],[36.0,1.0,20.0,6.0,30.0,3.4,2473.0,1.0,0.2330756...|
|(19,[0,5,8,9,12,13,14,15,16,18],[50.0,1.0,9.0,7.0,30.0,3.4,175729.0,3.0,3.0,0...|
+--------------------------------------------------------------------------------+
only showing top 3 rows



### Step 5 — Feature Scaling

`StandardScaler` standardizes the assembled feature vector to zero mean and unit variance. Features with large ranges (`author_post_count` can reach millions, `selftext_len` can reach thousands) would otherwise dominate binary features like `has_question` and `is_known_bot` which only take values 0 or 1. Scaling ensures all features contribute proportionally to the model.

The scaler is fit on training data only and applied to all three splits. Fitting on the full dataset would leak validation and test set statistics into training. We therefore perform the train/test split before fitting the scaler.

In [29]:
# ── TRAIN/VAL/TEST SPLIT ──────────────────────────────────────────────────────
from pyspark.ml.feature import StandardScaler

# Time-based split boundaries as Unix timestamps
TRAIN_END = 1483228800  # 2017-01-01 00:00:00 UTC
VAL_END   = 1514764800  # 2018-01-01 00:00:00 UTC

train_df = df_features.filter(F.col("created_utc") < TRAIN_END)
val_df   = df_features.filter(
    (F.col("created_utc") >= TRAIN_END) &
    (F.col("created_utc") < VAL_END)
)
test_df  = df_features.filter(F.col("created_utc") >= VAL_END)

print(f"Train rows: {train_df.count():,}")
print(f"Val rows:   {val_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

# Fit scaler on training data only — prevents leakage from val/test sets
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(train_df)

# Apply fitted scaler to all three splits
train_df = scaler_model.transform(train_df)
val_df   = scaler_model.transform(val_df)
test_df  = scaler_model.transform(test_df)

print("\nScaling complete.")
print(f"Train rows: {train_df.count():,}")
print(f"Val rows:   {val_df.count():,}")
print(f"Test rows:  {test_df.count():,}")
train_df.select("features").show(3, truncate=80)

Train rows: 276,442,594
Val rows:   114,089,205
Test rows:  144,949,019

Scaling complete.
Train rows: 276,442,594
Val rows:   114,089,205
Test rows:  144,949,019
+--------------------------------------------------------------------------------+
|                                                                        features|
+--------------------------------------------------------------------------------+
|[-0.3794727388086728,0.0,0.0,1.8308506054916096,-0.07351652556800245,0.046569...|
|[-0.3794727388086728,0.0,0.0,-0.5461942079725822,-0.07351652556800245,0.04656...|
|[0.01009276171008487,0.0,0.0,-0.5461942079725822,-0.07351652556800245,0.04656...|
+--------------------------------------------------------------------------------+
only showing top 3 rows



### Scaling and Split Results

Time-based train/val/test split:

| Split | Rows | % of Total |
|---|---|---|
| Train (2012–2016) | 276,442,594 | 51.6% |
| Validation (2017) | 114,089,205 | 21.3% |
| Test (2018) | 144,949,019 | 27.1% |

The scaled feature vectors show standardized values centered around 0 with values both positive and negative, confirming `StandardScaler` is working correctly. The split proportions are reasonable with over half the data in training, with a full year each for validation and test.

### Persist Train/Val/Test Splits

We persist the three splits to disk before model training to break the lineage chain (features_nlp → imputer → StringIndexer → VectorAssembler → StandardScaler → time filter). Without persistence, every action during model training would re-execute this entire chain. Reading from persisted Parquet eliminates this overhead and significantly reduces training time.

In [30]:
# ── PERSIST TRAIN/VAL/TEST SPLITS ────────────────────────────────────────────

TRAIN_DIR = "../data/processed/train"
VAL_DIR   = "../data/processed/val"
TEST_DIR  = "../data/processed/test"

print("Writing train split...")
train_df.write.mode("overwrite").parquet(TRAIN_DIR)
print("Writing val split...")
val_df.write.mode("overwrite").parquet(VAL_DIR)
print("Writing test split...")
test_df.write.mode("overwrite").parquet(TEST_DIR)

print("\nReloading splits from disk...")
train_df = spark.read.parquet(TRAIN_DIR)
val_df   = spark.read.parquet(VAL_DIR)
test_df  = spark.read.parquet(TEST_DIR)

print(f"Train rows: {train_df.count():,}")
print(f"Val rows:   {val_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

Writing train split...
Writing val split...
Writing test split...

Reloading splits from disk...
Train rows: 276,442,594
Val rows:   114,089,205
Test rows:  144,949,019


In [4]:
# Reloading train val and test dfs
# Note: This is checkpoint for model fitting; data load is done in persist cell
#       to validate counts

TRAIN_DIR = "../data/processed/train"
VAL_DIR   = "../data/processed/val"
TEST_DIR  = "../data/processed/test"


print("\nReloading splits from disk...")
train_df = spark.read.parquet(TRAIN_DIR)
val_df   = spark.read.parquet(VAL_DIR)
test_df  = spark.read.parquet(TEST_DIR)

print(f"Train rows: {train_df.count():,}")
print(f"Val rows:   {val_df.count():,}")
print(f"Test rows:  {test_df.count():,}")


Reloading splits from disk...
Train rows: 276,442,594
Val rows:   114,089,205
Test rows:  144,949,019


## Milestone 3: Model Training

We use `SparkXGBClassifier` from the `xgboost.spark` module, which ships with XGBoost ≥ 1.6. The cell below verifies availability and installs if needed.

In [5]:
# Check for XGBoost and Install if needed

try:
    from xgboost.spark import SparkXGBClassifier
    import xgboost as xgb
    print(f"XGBoost {xgb.__version__} available — SparkXGBClassifier ready")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost>=1.7"])
    from xgboost.spark import SparkXGBClassifier
    import xgboost as xgb
    print(f"Installed XGBoost {xgb.__version__}")

XGBoost 2.0.3 available — SparkXGBClassifier ready


### Class Weight Computation

To address label imbalance (low-engagement 44.3%, viral 29.9%, crowd-pleaser 14.0%, debate-starter 11.8%), we compute inverse-frequency weights from the training set only, then broadcast and join to all three splits. This prevents information leakage from val/test distributions into the weight values.

Weights are computed as `N / (num_classes * class_count)`, the normalized inverse frequency formulation used by scikit-learn, which keeps the effective total sample weight equal to N regardless of class distribution.

In [6]:
from pyspark.sql import functions as F

NUM_CLASSES_4 = 4
NUM_CLASSES_2 = 2

N_train = train_df.count()

# Compute per-class counts from train only
class_counts_4 = (
    train_df.groupBy("label_4class_idx")
            .agg(F.count("*").alias("class_count"))
)

class_counts_2 = (
    train_df.groupBy("label_binary_idx")
            .agg(F.count("*").alias("class_count"))
)

# Normalize: weight = N / (num_classes * class_count)
weight_map_4 = (
    class_counts_4
    .withColumn("weight_4class", F.lit(N_train) / (F.lit(NUM_CLASSES_4) * F.col("class_count")))
    .select("label_4class_idx", "weight_4class")
)

weight_map_2 = (
    class_counts_2
    .withColumn("weight_binary", F.lit(N_train) / (F.lit(NUM_CLASSES_2) * F.col("class_count")))
    .select("label_binary_idx", "weight_binary")
)

weight_map_4.orderBy("label_4class_idx").show()
weight_map_2.orderBy("label_binary_idx").show()

+----------------+------------------+
|label_4class_idx|     weight_4class|
+----------------+------------------+
|             0.0|0.5301268182465868|
|             1.0|0.9731065704183982|
|             2.0|1.6904547488515607|
|             3.0| 2.022386046668015|
+----------------+------------------+

+----------------+------------------+
|label_binary_idx|     weight_binary|
+----------------+------------------+
|             0.0|0.9462264655073706|
|             1.0|1.0602536364931736|
+----------------+------------------+



### Class Weight Results

The 4-class weights reflect the label imbalance introduced by the per-subreddit 75th-percentile thresholding scheme. Low-engagement (class 0) receives the smallest weight (0.530) because it is the majority class at 44.3% of the training set. Viral (class 1) is nearly balanced at 0.973, while crowd-pleaser (class 2, 14.0%) and debate-starter (class 3, 11.8%) receive the largest upweights (1.690 and 2.022 respectively) to compensate for their underrepresentation. The binary weights are close to 1.0 for both classes (0.946 and 1.060), consistent with a near-balanced binary split of 44.3% / 55.7% — the binary task requires minimal correction.

These weights will be passed to `SparkXGBClassifier` via `weight_col` so that the boosting objective penalizes misclassification of minority classes proportionally during tree construction.

### Joining Class Weights to Splits

We broadcast-join the 4-class and binary weight maps onto all three splits. The join is essentially free, both weight maps are 4 rows and 2 rows respectively, so Spark resolves them as broadcast hash joins with no shuffle.

After joining, we project each split down to only the three columns XGBoost needs (`features`, `label_4class_idx`, `weight_4class`). This avoids loading the full wide DataFrame which preserves all original columns including raw text and metadata into the XGBoost DMatrix during training, significantly reducing memory pressure on the JVM. The full-width `train_w`, `val_w`, and `test_w` are retained as lazy DataFrames for any post-hoc analysis in evaluation.

In [9]:
def attach_weights(df, wmap_4, wmap_2):
    return (
        df
        .join(F.broadcast(wmap_4), on="label_4class_idx", how="left")
        .join(F.broadcast(wmap_2), on="label_binary_idx", how="left")
    )

train_w = attach_weights(train_df, weight_map_4, weight_map_2)
val_w   = attach_weights(val_df,   weight_map_4, weight_map_2)
test_w  = attach_weights(test_df,  weight_map_4, weight_map_2)

# Project down to only what XGBoost needs — avoids loading wide DataFrame into DMatrix
train_xgb = train_w.select("features", "label_4class_idx", "weight_4class").repartition(16)
val_xgb   = val_w.select("features", "label_4class_idx", "weight_4class").repartition(16)
test_xgb  = test_w.select("features", "label_4class_idx", "weight_4class").repartition(16)

print("Weight join complete. Projected splits ready for training.")

Weight join complete. Projected splits ready for training.


In [10]:
print(f"train_xgb partitions: {train_xgb.rdd.getNumPartitions()}")
print(f"val_xgb partitions:   {val_xgb.rdd.getNumPartitions()}")
print(f"test_xgb partitions:  {test_xgb.rdd.getNumPartitions()}")

train_xgb partitions: 16
val_xgb partitions:   16
test_xgb partitions:  16


### XGBoost Model Configurations

We train two configurations on the 4-class problem to analyze the underfitting/overfitting tradeoff.

**Config A — Conservative (shallow, regularized):** `max_depth=4`, low learning rate, strong L2 regularization. Prioritizes stability; expected to underfit slightly given the complexity of 19 features across 102K subreddits.

**Config B — Aggressive (deep, fast learning):** `max_depth=8`, higher learning rate, light regularization. Risks overfitting on subreddit-level aggregation features that may encode training-period-specific behavior given the temporal train/val/test split.

Both configs target the 4-class label. `SparkXGBClassifier` sets the multiclass objective internally and uses `mlogloss` as the default evaluation metric — neither is passed explicitly. To avoid JVM OOM in `local[*]` mode, we use `num_workers=1` with `nthread=16` so XGBoost runs as a single worker using all 16 cores via multithreaded tree-building rather than 16 concurrent booster instances.

In [11]:
COMMON_PARAMS = dict(
    features_col="features",
    label_col="label_4class_idx",
    weight_col="weight_4class",
    num_class=4,
    num_workers=16, # Match partition size / core config
    device="cpu",
    seed=42,
)

# Config A: conservative — shallow trees, strong regularization
config_A = SparkXGBClassifier(
    **COMMON_PARAMS,
    max_depth=4,
    n_estimators=200,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    reg_alpha=1.0,
    min_child_weight=50,
)

# Config B: aggressive — deep trees, faster learning, light regularization
config_B = SparkXGBClassifier(
    **COMMON_PARAMS,
    max_depth=8,
    n_estimators=300,
    learning_rate=0.1,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=10,
)

print("Config A and Config B defined.")

Config A and Config B defined.


### Training Config A (Conservative)

SparkXGBClassifier distributes tree-building across 16 Spark workers, each processing ~17M rows in parallel within the single JVM. Open the Spark UI at port 4040 during training to capture the parallelism screenshot required by the milestone.

In [12]:
# Create directories

import os
os.makedirs("../outputs/models/xgb_config_A_checkpoints", exist_ok=True)
os.makedirs("../outputs/models", exist_ok=True)

In [14]:
import time

print("Training Config A...")
t0 = time.time()

model_A = config_A.fit(train_xgb)
elapsed_A = time.time() - t0
print(f"Config A training time: {elapsed_A:.1f}s")

model_A.write().overwrite().save("../outputs/models/xgb_config_A")
print("Config A saved.")

Training Config A...


2026-05-09 01:33:24,408 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 16 workers with
	booster params: {'objective': 'multi:softprob', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 50, 'reg_alpha': 1.0, 'reg_lambda': 5.0, 'subsample': 0.8, 'num_class': 4, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 200}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-09 02:32:05,254 INFO XGBoost-PySpark: _fit Finished xgboost training!


Config A training time: 3610.9s
Config A saved.


#### Spark UI — Executor Summary

Querying the Spark REST API to document executor configuration and confirm distributed execution for Config A.

In [15]:
import requests
import pandas as pd

sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

executors = requests.get(url).json()

sparkUI_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
sparkUI_df['maxMemory_GB'] = (sparkUI_df['maxMemory'] / (1024**3)).round(2)
print(sparkUI_df)
print(f"\nSpark master: {sc.master}")

       id  totalCores    maxMemory  activeTasks  isActive  maxMemory_GB
0  driver          16  77120667648            0      True         71.82

Spark master: local[*]


### Training Config B (Aggressive)

Same distributed setup as Config A w/ 16 workers each processing ~17M rows. Deeper trees and faster learning rate are expected to produce a larger model with longer training time.

In [16]:
import time

print("Training Config B...")
t0 = time.time()

model_B = config_B.fit(train_xgb)
elapsed_B = time.time() - t0
print(f"Config B training time: {elapsed_B:.1f}s")

model_B.write().overwrite().save("../outputs/models/xgb_config_B")
print("Config B saved.")

Training Config B...


2026-05-09 03:47:32,161 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 16 workers with
	booster params: {'objective': 'multi:softprob', 'colsample_bytree': 0.7, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 10, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.7, 'num_class': 4, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-09 05:43:11,075 INFO XGBoost-PySpark: _fit Finished xgboost training!


Config B training time: 7059.0s
Config B saved.
